<a href="https://colab.research.google.com/github/SrijaMadarapu8/AI-ML/blob/main/MCP_Tool_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MCP Tool Server + LangChain/LangGraph Agent (OpenAI-Powered)**

A minimal but complete demonstration of building a custom Model Context Protocol
(MCP) server, exposing it as tools to a frontier model, and orchestrating tool
calls through a LangGraph ReAct agent run entirely in Google Colab.


**1. What This Project Demonstrates**


*   Building an MCP server from scratch using FastMCP
*   Exposing custom tools (a resume/project lookup tool, plus a simple math tool) with auto-generated schemas
*   Connecting a LangChain/LangGraph agent to that server over the MCP stdio transport
*   Wiring the agent to a frontier model (OpenAI, via langchain-openai)
*   Understanding and being able to explain the three-stage tool-calling loop: binding schemas → model requests a tool call → client executes and returns the result
*   Real debugging of environment-specific issues encountered when running MCP's subprocess-based transport inside a notebook kernel

Key point: The model never executes any code. It only receives tool
descriptions and, when relevant, returns a structured request naming which
tool to call and with what arguments. The client (this notebook) is
responsible for actually running the tool and sending the result back.

**2. Tech Stack**

*   Language : Python 3.12
*   Notebook environment : Google ColabReal
*   MCP SDK : mcp (official Python SDK)
*   Tool server framework : FastMCP (part of mcp)
*   Agent orchestration : langchain, langgraph
*   MCP↔LangChain bridge : langchain-mcp-adapters
*   Model provider : langchain-openai

**3. Setup Steps**


3.1 Install dependencies

In [1]:
!pip install mcp langchain langchain-openai langchain-mcp-adapters langgraph python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.7 MB/s eta 0:00:00


3.2 Load the key into the environment

In [14]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

3.3 Write the MCP server file

In [34]:
%%writefile server.py
from typing import Union
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("resume-tools")

# --- Math tools ---

@mcp.tool()
def add(a: Union[int, float], b: Union[int, float]) -> Union[int, float]:
    """Add two numbers."""
    return a + b

@mcp.tool()
def multiply(a: Union[int, float], b: Union[int, float]) -> Union[int, float]:
    """Multiply two numbers."""
    return a * b

@mcp.tool()
def division(a: Union[int, float], b: Union[int, float]) -> Union[int, float]:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

@mcp.tool()
def square_root(a: Union[int, float]) -> Union[int, float]:
    """Compute the square root of a number."""
    if a < 0:
        raise ValueError("Cannot compute square root of a negative number.")
    return a ** 0.5

# --- Resume/project tools ---

@mcp.tool()
def get_project_details(project_name: str) -> str:
    """Get details about one of my resume projects by key name."""
    projects = {
        "robotic_arm_rl": "Trained a UFactory X-Arm 7 to perform pick-and-place using PPO reinforcement learning in a MuJoCo digital twin, parallelized across 512+ simulated environments on AMD ROCm hardware.",
        "rag_pipeline": "Built a Retrieval-Augmented Generation pipeline for document understanding using ChromaDB, OpenAI embeddings, and LangChain, grounding LLM responses in retrieved document chunks.",
        "cv_reconstruction": "Built a 3D reconstruction pipeline using SIFT, RANSAC, and Structure-from-Motion, plus Gaussian Splatting and NeRF for photorealistic scene rendering, reaching 95% feature-matching accuracy."
    }
    return projects.get(project_name, f"No project found for '{project_name}'. Available: {list(projects.keys())}")

if __name__ == "__main__":
    mcp.run()

Overwriting server.py


**4. Enhance the LLM with Tools**

4.1 Imports and model

In [35]:
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_core.messages import HumanMessage, ToolMessage

model = ChatOpenAI(
    model="gpt-5.4-nano",
    base_url="https://agentate-model.services.ai.azure.com/openai/v1",
    api_key=os.environ["OPENAI_API_KEY"]
)

 4.2 List available tools

In [36]:
log_file = open("server_stderr.log", "w")

server_params = StdioServerParameters(command="python", args=["server.py"])

async with stdio_client(server_params, errlog=log_file) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)
        print("Available tools:")
        for tool in tools:
            print(f"{tool}\n")

Available tools:
name='add' description='Add two numbers.' args_schema={'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'A'}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'B'}}, 'required': ['a', 'b'], 'title': 'addArguments', 'type': 'object'} handle_tool_error=<function _handle_mcp_tool_error at 0x7869de5219e0> response_format='content_and_artifact' coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7869daef1300>

name='multiply' description='Multiply two numbers.' args_schema={'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'A'}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}], 'title': 'B'}}, 'required': ['a', 'b'], 'title': 'multiplyArguments', 'type': 'object'} handle_tool_error=<function _handle_mcp_tool_error at 0x7869de5219e0> response_format='content_and_artifact' coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7869daef0

 4.3 Agent function

In [18]:
async def run_react_agent(query: str):
    async with stdio_client(server_params, errlog=log_file) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            agent = create_agent(model, tools)
            agent_response = await agent.ainvoke({"messages": query})
            return agent_response

4.4 Run test queries

In [19]:
try:
    query = "what is the square root of 55?"
    agent_response = await run_react_agent(query)
    for m in agent_response['messages']:
        m.pretty_print()
except* Exception as eg:
    for e in eg.exceptions:
        import traceback
        traceback.print_exception(type(e), e, e.__traceback__)

================================ Human Message =================================

what is the square root of 55?
================================== Ai Message ==================================
Tool Calls:
  square_root (call_pTiOloc7FXBFf2gNg6pprAjk)
 Call ID: call_pTiOloc7FXBFf2gNg6pprAjk
  Args:
    number: 55
================================= Tool Message =================================
Name: square_root

[{'type': 'text', 'text': '7.416198487095663', 'id': 'lc_f2f5fcff-f13b-433e-9861-59456e20b939'}]
================================== Ai Message ==================================

The square root of 55 is approximately **7.4162**.


4.5 Bind Tools to the LLM

In [37]:
async with stdio_client(server_params, errlog=log_file) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)

        model_with_tools = model.bind_tools(tools)

        bound_tools = model_with_tools.kwargs['tools']
        for t in bound_tools:
            func = t['function']
            print(f"Tool: {func['name']}")
            print(f"  Description: {func['description']}")
            print(f"  Parameters: {func['parameters']}\n")

Tool: add
  Description: Add two numbers.
  Parameters: {'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}}, 'required': ['a', 'b'], 'type': 'object'}

Tool: multiply
  Description: Multiply two numbers.
  Parameters: {'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}}, 'required': ['a', 'b'], 'type': 'object'}

Tool: division
  Description: Divide a by b.
  Parameters: {'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}, 'b': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}}, 'required': ['a', 'b'], 'type': 'object'}

Tool: square_root
  Description: Compute the square root of a number.
  Parameters: {'properties': {'a': {'anyOf': [{'type': 'integer'}, {'type': 'number'}]}}, 'required': ['a'], 'type': 'object'}

Tool: get_project_details
  Description: Get details about one of my resume projects by

4.5 The LLM Decides Which Tool to Call

In [23]:
async with stdio_client(server_params, errlog=log_file) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)
        model_with_tools = model.bind_tools(tools)
        messages = [HumanMessage(content="what is the square root of 55?")]
        ai_response = await model_with_tools.ainvoke(messages)
        print("Content:", ai_response.content)
        print("\nTool calls requested by the LLM:")
        for tc in ai_response.tool_calls:
            print(f"  Function: {tc['name']}, Arguments: {tc['args']}")

Content: 

Tool calls requested by the LLM:
  Function: square_root, Arguments: {'number': 55}


4.7 Execute the Tool and Pass the Result Back

In [26]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def mcp_session():
    async with stdio_client(server_params, errlog=log_file) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            yield session

In [27]:
async with mcp_session() as session:
    tools = await load_mcp_tools(session)
    model_with_tools = model.bind_tools(tools)
    messages = [HumanMessage(content="what is the square root of 55?")]
    ai_response = await model_with_tools.ainvoke(messages)
    messages.append(ai_response)

    for tool_call in ai_response.tool_calls:
        result = await session.call_tool(tool_call['name'], arguments=tool_call['args'])
        print(f"Tool executed: {tool_call['name']}({tool_call['args']}) → {result}")
        messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    final_response = await model_with_tools.ainvoke(messages)
    print(f"\nFinal LLM answer: {final_response.content}")

Tool executed: square_root({'number': 55}) → meta=None content=[TextContent(type='text', text='7.416198487095663', annotations=None, meta=None)] structuredContent={'result': 7.416198487095663} isError=False

Final LLM answer: The square root of 55 is approximately **7.4162**.
